In [118]:
!pip install -q gradio plotly ta google-genai kagglehub

In [119]:
import kagglehub

path = kagglehub.dataset_download(
    "soachishti/pakistan-stock-exchange-psx-complete-dataset"
)

print(path)

Using Colab cache for faster access to the 'pakistan-stock-exchange-psx-complete-dataset' dataset.
/kaggle/input/pakistan-stock-exchange-psx-complete-dataset


In [120]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import gradio as gr

from ta.trend import SMAIndicator
from ta.momentum import RSIIndicator
from ta.trend import MACD

from google import genai

In [121]:
import glob

csv_files = glob.glob(f"{path}/**/*.csv", recursive=True)

print(csv_files)

['/kaggle/input/pakistan-stock-exchange-psx-complete-dataset/data/lot.csv', '/kaggle/input/pakistan-stock-exchange-psx-complete-dataset/data/sector.csv', '/kaggle/input/pakistan-stock-exchange-psx-complete-dataset/data/historical/mtl.csv', '/kaggle/input/pakistan-stock-exchange-psx-complete-dataset/data/historical/efug.csv', '/kaggle/input/pakistan-stock-exchange-psx-complete-dataset/data/historical/pael.csv', '/kaggle/input/pakistan-stock-exchange-psx-complete-dataset/data/historical/macfl.csv', '/kaggle/input/pakistan-stock-exchange-psx-complete-dataset/data/historical/snai.csv', '/kaggle/input/pakistan-stock-exchange-psx-complete-dataset/data/historical/jksm.csv', '/kaggle/input/pakistan-stock-exchange-psx-complete-dataset/data/historical/meht.csv', '/kaggle/input/pakistan-stock-exchange-psx-complete-dataset/data/historical/tplt.csv', '/kaggle/input/pakistan-stock-exchange-psx-complete-dataset/data/historical/ppp.csv', '/kaggle/input/pakistan-stock-exchange-psx-complete-dataset/data

In [122]:
import os

for root, dirs, files in os.walk(path):
    print(root)
    print(files)
    print("-"*50)

/kaggle/input/pakistan-stock-exchange-psx-complete-dataset
[]
--------------------------------------------------
/kaggle/input/pakistan-stock-exchange-psx-complete-dataset/data
['lot.csv', 'sector.csv']
--------------------------------------------------
/kaggle/input/pakistan-stock-exchange-psx-complete-dataset/data/historical
['mtl.csv', 'efug.csv', 'pael.csv', 'macfl.csv', 'snai.csv', 'jksm.csv', 'meht.csv', 'tplt.csv', 'ppp.csv', 'pcal.csv', 'pkgp.csv', 'esbl.csv', 'ghnl.csv', 'zahid.csv', 'tpli.csv', 'dawh.csv', 'jsgcl.csv', 'ssom.csv', 'ael.csv', 'treet.csv', 'asc.csv', 'kasbm.csv', 'nml.csv', 'kohc.csv', 'dfml.csv', 'nrsl.csv', 'colg.csv', 'bapl.csv', 'tele.csv', 'npl.csv', 'olpl.csv', 'epcl.csv', 'power.csv', 'agl.csv', 'nsrm.csv', 'nicl.csv', 'cwsm.csv', 'kel.csv', 'chas.csv', 'mtil.csv', 'surc.csv', 'exide.csv', 'icl.csv', 'pibtl.csv', 'kml.csv', 'hcar.csv', 'siem.csv', 'dwtm.csv', 'isil.csv', 'aicl.csv', 'ffl.csv', 'rewm.csv', 'kohp.csv', 'koil.csv', 'mdtl.csv', 'idsm.csv', '

In [123]:
import pandas as pd

lots_path = f"{path}/data/lot.csv"
print (lots_path)

lots_df = pd.read_csv(lots_path)

print(lots_df.head())

print(lots_df.columns)

/kaggle/input/pakistan-stock-exchange-psx-complete-dataset/data/lot.csv
  market security_ticker  lot
0    REG             786  500
1    REG            AABS  100
2    REG             AAL   50
3    REG            AASM  500
4    REG            AATM  500
Index(['market', 'security_ticker', 'lot'], dtype='object')


In [124]:
lots_df.columns = [
    c.lower().strip()
    for c in lots_df.columns
]

print(lots_df.columns)
symbols = sorted(
    lots_df['security_ticker']
    .dropna()
    .unique()
    .tolist()
)

print(symbols[:20])
historical_path = f"{path}/data/historical"

print(os.listdir(historical_path)[:10])

Index(['market', 'security_ticker', 'lot'], dtype='object')
['786', 'AABS', 'AAL', 'AASM', 'AATM', 'ABL', 'ABOT', 'ABSON', 'ACPL', 'ADAMS', 'ADMM', 'ADOS', 'AEL', 'AGIC', 'AGIL', 'AGL', 'AGLNCPS', 'AGP', 'AGSML', 'AGTL']
['mtl.csv', 'efug.csv', 'pael.csv', 'macfl.csv', 'snai.csv', 'jksm.csv', 'meht.csv', 'tplt.csv', 'ppp.csv', 'pcal.csv']


In [125]:
import os

def load_stock_data(symbol):

    file_path = os.path.join(
        historical_path,
        f"{symbol.lower()}.csv"
    )
    #print ("Stock data file path: ",file_path)

    if not os.path.exists(file_path):
        return None

    stock_df = pd.read_csv(file_path)

    stock_df.columns = [
        c.lower().strip()
        for c in stock_df.columns
    ]

    # Detect date column automatically
    # for col in stock_df.columns:
    #     if 'time' in col:
    #         date_col = col
    #         break
    date_col = 'time'

    stock_df[date_col] = pd.to_datetime(
        stock_df[date_col]
    )

    stock_df = stock_df.sort_values(date_col)

    return stock_df

In [126]:
test_df = load_stock_data("EPCL")

print(test_df.head())

print(test_df.columns)

           time   open   high    low  close   volume
2978 2008-07-21  23.52  24.50  22.50  23.31  1631000
2977 2008-07-22  23.31  24.47  23.27  24.47  1211500
2976 2008-07-23  24.47  25.69  24.80  25.69  1303000
2975 2008-07-24  25.69  26.97  25.00  26.97  2363500
2974 2008-07-25  26.97  28.31  27.10  27.69  2251500
Index(['time', 'open', 'high', 'low', 'close', 'volume'], dtype='object')


In [127]:
from ta.trend import SMAIndicator
from ta.momentum import RSIIndicator
from ta.trend import MACD

def add_indicators(stock_df):

    stock_df = stock_df.copy()

    close_col = 'close'

    stock_df['sma20'] = SMAIndicator(
        close=stock_df[close_col],
        window=20
    ).sma_indicator()

    stock_df['sma50'] = SMAIndicator(
        close=stock_df[close_col],
        window=50
    ).sma_indicator()

    stock_df['rsi'] = RSIIndicator(
        close=stock_df[close_col],
        window=14
    ).rsi()

    macd = MACD(stock_df[close_col])

    stock_df['macd'] = macd.macd()

    stock_df['macd_signal'] = macd.macd_signal()

    return stock_df

In [128]:
def create_candlestick(stock_df):

    fig = go.Figure()

    fig.add_trace(
        go.Candlestick(
            x=stock_df['time'],
            open=stock_df['open'],
            high=stock_df['high'],
            low=stock_df['low'],
            close=stock_df['close'],
            name='Price'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=stock_df['time'],
            y=stock_df['sma20'],
            mode='lines',
            name='SMA20'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=stock_df['time'],
            y=stock_df['sma50'],
            mode='lines',
            name='SMA50'
        )
    )

    fig.update_layout(
        title="Price Analysis",
        xaxis_rangeslider_visible=False,
        height=600
    )

    return fig

In [129]:
def create_volume_chart(stock_df):

    fig = px.bar(
        stock_df,
        x='time',
        y='volume',
        title='Volume Analysis'
    )

    return fig

In [130]:
def create_rsi_chart(stock_df):

    fig = px.line(
        stock_df,
        x='time',
        y='rsi',
        title='RSI Indicator'
    )

    return fig

In [131]:
from google import genai
import os
from google.colab import userdata


os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
client = genai.Client()
def generate_ai_analysis(symbol, stock_df):

    latest = stock_df.iloc[-1]

    prompt = f"""
    Analyze this Pakistan stock.

    Symbol: {symbol}

    Latest Close: {latest['close']}

    RSI: {latest['rsi']}

    MACD: {latest['macd']}

    Give:
    1. Trend analysis
    2. Risk level
    3. Momentum
    4. Technical interpretation
    5. Long term outlook
    """

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

In [132]:
def analyze_stock(symbol):

    stock_df = load_stock_data(symbol)

    if stock_df is None:
        return (
            "Stock file not found",
            None,
            None,
            None,
            "No AI analysis"
        )

    stock_df = add_indicators(stock_df)

    candle = create_candlestick(stock_df)

    volume = create_volume_chart(stock_df)

    rsi = create_rsi_chart(stock_df)

    ai_analysis = generate_ai_analysis(
        symbol,
        stock_df
    )

    latest = stock_df.iloc[-1]

    summary = f"""
    Symbol: {symbol}

    Latest Close: {latest['close']}

    Latest Volume: {latest['volume']}
    """

    return (
        summary,
        candle,
        volume,
        rsi,
        ai_analysis
    )

In [133]:
import pandas as pd

sector_df = pd.read_csv(f"{path}/data/sector.csv")

sector_df.columns = sector_df.columns.str.lower().str.strip()

print(sector_df.head())
print(sector_df.columns)

sector_df = sector_df.rename(columns={
    "security_ticker": "symbol"
})


   security_id security_ticker                 security_name  sector_id  \
0        21490            DOMF   Dominion Stock Fund Limited        806   
1        21900            HGFA               HBL Growth Fund        806   
2        21902            INMF  Investec Mutual Fund Limited        806   
3        21903            HIFA           HBL Investment Fund        806   
4        21913           SINDM                Sindh Modaraba        819   

                    sector  
0  CLOSE - END MUTUAL FUND  
1  CLOSE - END MUTUAL FUND  
2  CLOSE - END MUTUAL FUND  
3  CLOSE - END MUTUAL FUND  
4                MODARABAS  
Index(['security_id', 'security_ticker', 'security_name', 'sector_id',
       'sector'],
      dtype='object')


In [134]:
def load_stock_with_sector(symbol):

    stock_df = load_stock_data(symbol)

    if stock_df is None:
        return None

    # attach sector info
    sector_info = sector_df[
        sector_df["symbol"] == symbol
    ]

    if len(sector_info) > 0:
        sector_map = sector_df.set_index("symbol")["sector"].to_dict()
        stock_df["sector"] = sector_map.get(symbol, "Unknown")
        stock_df["sector"] = sector_info.iloc[0]["sector"]
        stock_df["security_name"] = sector_info.iloc[0]["security_name"]
    else:
        stock_df["sector"] = "Unknown"
        stock_df["security_name"] = symbol

    return stock_df

In [135]:
def sector_analysis():

    all_data = []

    for symbol in sector_df["symbol"].unique():

        stock_df = load_stock_data(symbol)

        if stock_df is None:
            continue

        stock_df["symbol"] = symbol

        # merge sector info
        sector_info = sector_df[
            sector_df["symbol"] == symbol
        ]

        if len(sector_info) > 0:
            stock_df["sector"] = sector_info.iloc[0]["sector"]
        else:
            stock_df["sector"] = "Unknown"

        all_data.append(stock_df.tail(1))  # latest row only

    combined = pd.concat(all_data)

    # now safe grouping
    sector_perf = combined.groupby("sector")["close"].mean().reset_index()

    fig = px.treemap(
        sector_perf,
        path=["sector"],
        values="close",
        color="close",
        title="PSX Sector Performance"
    )

    return fig

In [136]:
import plotly.graph_objects as go

def compare_stocks(sym1, sym2):

    # Load stock files
    s1 = load_stock_data(sym1)
    s2 = load_stock_data(sym2)

    if s1 is None or s2 is None:
        return None

    # Normalize columns
    s1.columns = s1.columns.str.lower().str.strip()
    s2.columns = s2.columns.str.lower().str.strip()

    # Detect date column automatically
    date_col_1 = None
    date_col_2 = None

    for col in s1.columns:
        if 'date' in col:
            date_col_1 = col
            break

    for col in s2.columns:
        if 'date' in col:
            date_col_2 = col
            break

    # Create chart
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=s1[date_col_1],
            y=s1['close'],
            mode='lines',
            name=sym1
        )
    )

    fig.add_trace(
        go.Scatter(
            x=s2[date_col_2],
            y=s2['close'],
            mode='lines',
            name=sym2
        )
    )

    fig.update_layout(
        title=f"{sym1} vs {sym2}",
        xaxis_title="Date",
        yaxis_title="Close Price",
        height=600
    )

    return fig

In [137]:
symbol_input = gr.Dropdown(
    choices=symbols,
    label="Select PSX Symbol",
    filterable=True   # optional in some versions
)

In [138]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# PSX AI Analytics Platform")

    with gr.Tab("Stock Analysis"):

        symbol_input = gr.Dropdown(
            choices=symbols,
            label="Select Symbol"
        )

        analyze_btn = gr.Button("Analyze")

        summary_output = gr.Textbox(label="Summary")

        candle_output = gr.Plot()

        volume_output = gr.Plot()

        rsi_output = gr.Plot()

        ai_output = gr.Markdown(label="AI Analysis")

        analyze_btn.click(
            analyze_stock,
            inputs=symbol_input,
            outputs=[
                summary_output,
                candle_output,
                volume_output,
                rsi_output,
                ai_output
            ]
        )

    print(df.columns)

    with gr.Tab("Sector Analysis"):

        sector_plot = gr.Plot(
            value=sector_analysis
        )

    with gr.Tab("Compare Stocks"):

        stock1 = gr.Dropdown(
            choices=symbols,
            label="Stock 1"
        )

        stock2 = gr.Dropdown(
            choices=symbols,
            label="Stock 2"
        )

        compare_btn = gr.Button("Compare")

        compare_plot = gr.Plot()

        compare_btn.click(
            compare_stocks,
            inputs=[stock1, stock2],
            outputs=compare_plot
        )

/tmp/ipykernel_3864/303448611.py:1: DeprecationWarning:

The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.



Index(['time', 'open', 'high', 'low', 'close', 'volume'], dtype='object')


In [139]:
demo.launch(share=True, debug=True)

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/indexes/base.py", line 3805, in get_loc
    return self._engine.get_loc(casted_key)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7081, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7089, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'symbol'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354,

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ec9ac49979d679d298.gradio.live
